In [ ]:
import requests, json
from urllib.parse import urljoin
import pandas as pd
from collections import Counter

In [ ]:
url = 'https://www.apicountries.com/countries'

response = requests.get(url).json()

df = pd.DataFrame(response)
cols = ['name'] + [col for col in df.columns if col != 'name']
df = df[cols]

# Case 1 Top 10 Country with Largest Area
largest_area = df.sort_values(by='area', ascending=False).reset_index(drop=True)
largest_area.head(10)

# Case 2 Most 10 Used Languages
most_languages = df['languages'].apply(
    lambda langs: [x['name'] for x in langs] if langs else []
).explode()

count = Counter(most_languages)
sort_count = sorted(count.items(), key=lambda x: (-x[1], x[0]))
print(sort_count[:10])

# Case 3 Number of Official Languages in Each Country
df['num_languages'] = df['languages'].apply(
    lambda x: len(x) if x else 0
)
df.sort_values(by=['num_languages', 'name'], ascending=[False, True])[['name', 'languages', 'num_languages']]

In [ ]:
# Scraping Pokemon Data with API
url = 'https://pokeapi.co/api/v2/pokemon/'

try:
    data = []
    for i in range(1, 1001):
        print(f'Pokemon number {i}')
        pokemon_url = urljoin(url, str(i))
        response = requests.get(pokemon_url, timeout=5)
        response.raise_for_status()
        pokemon_details = response.json()
        data.append({
            'pokemonId': pokemon_details['id'],
            'pokemonName': pokemon_details['name'],
            "pokemonType": [
                t["type"]["name"]
                for t in pokemon_details["types"]
            ],
            'pokemonHeight': pokemon_details['height'],
            'pokemonWeight': pokemon_details['weight'],
            'pokemonBaseExperience': pokemon_details['base_experience']
        })
    with open('pokemon.json', 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4)
        print('API Data Saved Succesfully')
    print('Data Saved Successfully')
except requests.exceptions.HTTPError:
    print('HTTP Error')
except requests.exceptions.Timeout:
    print('Time Out')
except Exception as e:
    print(f'Error: {e}')